# 🎙️ Kashmiri Speech-to-Text with Whisper Large-V3-Turbo (Google Colab)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sameerahmad86232/cipher/blob/main/colab-kashmiri-whisper-speech-to-text.ipynb)

Record or upload your voice speaking Kashmiri and transcribe it into authentic **Kashmiri Perso-Arabic script** with our fine-tuned **OpenAI Whisper Large-V3-Turbo** model (`Sameer0313/kashmiri-whisper-lora`, CER: 22.42%).

**Hardware Recommendation:** Runtime -> Change runtime type -> **T4 GPU**.

In [ ]:
# Step 1: Install Dependencies
!pip install -q transformers peft soundfile librosa evaluate jiwer huggingface_hub accelerate

In [ ]:
# Step 2: Load Kashmiri Whisper Large-V3-Turbo Model
import torch
from transformers import WhisperProcessor, WhisperForConditionalGeneration
from peft import PeftModel

BASE_MODEL = "openai/whisper-large-v3-turbo"
ADAPTER_REPO = "Sameer0313/kashmiri-whisper-lora"

print("[*] Loading Whisper processor...")
processor = WhisperProcessor.from_pretrained(BASE_MODEL, language="urdu", task="transcribe")

print("[*] Loading base model and attaching fine-tuned Kashmiri adapter...")
base_model = WhisperForConditionalGeneration.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto"
)
model = PeftModel.from_pretrained(base_model, ADAPTER_REPO)
model.eval()
print("[+] Kashmiri Whisper Model loaded successfully!")

In [ ]:
# Step 3: Interactive In-Browser Microphone Recorder
from IPython.display import HTML, Audio, display
from google.colab import output
import base64

RECORD_HTML = """
<div style="border: 2px solid #3b82f6; border-radius: 12px; padding: 20px; text-align: center; max-width: 450px; background: #1e293b; color: white;">
  <h3 style="margin-top: 0;">🎤 Record Kashmiri Speech</h3>
  <p>Click <b>Start Recording</b>, speak a sentence in Kashmiri, then click <b>Stop</b>.</p>
  <button id="startBtn" style="background: #10b981; color: white; padding: 10px 20px; border: none; border-radius: 8px; font-weight: bold; cursor: pointer; margin-right: 10px;">Start Recording</button>
  <button id="stopBtn" disabled style="background: #ef4444; color: white; padding: 10px 20px; border: none; border-radius: 8px; font-weight: bold; cursor: pointer;">Stop Recording</button>
  <div id="status" style="margin-top: 15px; color: #94a3b8;">Ready</div>
</div>

<script>
var mediaRecorder;
var audioChunks = [];

document.getElementById('startBtn').onclick = async () => {
  audioChunks = [];
  const stream = await navigator.mediaDevices.getUserMedia({ audio: true });
  mediaRecorder = new MediaRecorder(stream);
  mediaRecorder.ondataavailable = e => audioChunks.push(e.data);
  mediaRecorder.onstop = async () => {
    const blob = new Blob(audioChunks, { type: 'audio/wav' });
    const reader = new FileReader();
    reader.readAsDataURL(blob);
    reader.onloadend = () => {
      google.colab.kernel.invokeFunction('save_audio', [reader.result], {});
    };
  };
  mediaRecorder.start();
  document.getElementById('startBtn').disabled = true;
  document.getElementById('stopBtn').disabled = false;
  document.getElementById('status').innerText = '🔴 Recording... Speak in Kashmiri!';
  document.getElementById('status').style.color = '#ef4444';
};

document.getElementById('stopBtn').onclick = () => {
  mediaRecorder.stop();
  document.getElementById('startBtn').disabled = false;
  document.getElementById('stopBtn').disabled = true;
  document.getElementById('status').innerText = '✅ Recording saved!';
  document.getElementById('status').style.color = '#10b981';
};
</script>
"""

def save_audio(data_uri):
    header, encoded = data_uri.split(",", 1)
    data = base64.b64decode(encoded)
    with open("/content/recorded_speech.wav", "wb") as f:
        f.write(data)
    print("\n[+] Audio recorded and saved to /content/recorded_speech.wav!")

output.register_callback('save_audio', save_audio)
display(HTML(RECORD_HTML))

In [ ]:
# Step 4: Transcribe Kashmiri Speech into Perso-Arabic Script!
import librosa
import soundfile as sf

AUDIO_FILE = "/content/recorded_speech.wav"

def transcribe_kashmiri_audio(path):
    speech, sr = librosa.load(path, sr=16000)
    inputs = processor(speech, sampling_rate=16000, return_tensors="pt").input_features
    inputs = inputs.to(model.device, dtype=model.dtype)
    
    with torch.no_grad():
        predicted_ids = model.generate(
            inputs,
            num_beams=2,
            max_new_tokens=225
        )
    return processor.batch_decode(predicted_ids, skip_special_tokens=True)[0]

try:
    transcription = transcribe_kashmiri_audio(AUDIO_FILE)
    print("=" * 60)
    print("✨ KASHMIRI TRANSCRIPTION (Perso-Arabic Script):")
    print(transcription)
    print("=" * 60)
except FileNotFoundError:
    print("[!] Please record an audio clip in Step 3 first, or upload your own .wav file to /content/")